# Best version of each model — shared comparison

Run this notebook from **24_10_2026_TianFeng**, top to bottom. It trains five
models from scratch and displays one shared metrics table with each author.
No other model file or checkpoint is needed. Running it does not write files.

Where a model family appeared more than once in the earlier comparison, keep
the variant with the highest **transition macro-F1**. This averages the six rare
transitions, which are the project's focus. Macro-F1, then accuracy, break exact
ties. The retained configurations are Yifei's logistic regression, linear SVM,
random forest and XGBoost, and Tianfeng's contextual MLP with posture memory.

These are the best **observed variants in the previous shared development-test
comparison**, not a claim that they are universally optimal. Minh's variants,
the MLP reference in Yifei's notebook and the plain contextual MLP did not rank
highest within their families. Their original files remain untouched. The main
run now fits only the five retained configurations, keeping their preprocessing
and model hyperparameters. The plain NN scores are still needed internally for
posture memory, but are not a separate model row.

**Libraries:** NumPy, pandas, scikit-learn, XGBoost, PyTorch and Matplotlib.
Install missing libraries in your notebook's Python environment. Package versions
and CPU thread settings can change scores. Optional tuning sections are off by
default; the main table is calculated from this run's predictions.

## 1. Imports

The models run on CPU. Import XGBoost before PyTorch and use one PyTorch CPU
thread to avoid the OpenMP conflict observed on this Mac. The original model
hyperparameters remain intact, including Yifei's two XGBoost worker threads.
This changes CPU scheduling only; matrix operations outside PyTorch use two
threads where supported.

In [1]:
import os
from copy import deepcopy

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(name, '2')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix
from threadpoolctl import threadpool_limits

threadpool_limits(limits=2, user_api='blas')
torch.set_num_threads(1)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 65)
np.set_printoptions(precision=4, suppress=True)
predictions_by_model = {}

## 2. Read the shared data

Only the 561 sensor-feature columns become predictors. Participant IDs define
context boundaries and validation groups; activity IDs are targets, not inputs.
The retained pipelines use float32 features. The shared metrics use labels 0–11;
Yifei's functions keep their original 1–12 labels internally and convert their
predictions once for the comparison.

One PyTorch CPU thread avoids the OpenMP conflict on this Mac. It can produce
slightly different predictions from the separate two-thread NN notebook, even
with the same seeds and hyperparameters. The table reports this run's scores.

In [2]:
train_df = pd.read_csv('../Minh/hapt_train.csv')
test_df = pd.read_csv('../Minh/hapt_test.csv')
feature_columns = train_df.columns[3:]
X = train_df[feature_columns].to_numpy(dtype=np.float32)
X_test = test_df[feature_columns].to_numpy(dtype=np.float32)
y = train_df.activity_id.to_numpy(dtype=np.int64) - 1
y_test = test_df.activity_id.to_numpy(dtype=np.int64) - 1
subject = train_df.subject_id.to_numpy()
test_subject = test_df.subject_id.to_numpy()
names = (train_df[['activity_id', 'activity_label']].drop_duplicates()
         .sort_values('activity_id').activity_label.tolist())

assert set(subject).isdisjoint(test_subject)
assert len(feature_columns) == 561
print(f'Train: {len(y)} windows, {len(np.unique(subject))} participants')
print(f'Test: {len(y_test)} windows, {len(np.unique(test_subject))} participants')
display(pd.DataFrame({'Activity': names, 'Train': np.bincount(y, minlength=12),
                      'Test': np.bincount(y_test, minlength=12)}))

Train: 7767 windows, 21 participants
Test: 3162 windows, 9 participants


,Activity,Train,Test
0,WALKING,1226,496
1,WALKING_UPSTAIRS,1073,471
2,WALKING_DOWNSTAIRS,987,420
3,SITTING,1293,508
4,STANDING,1423,556
5,LAYING,1413,545
6,STAND_TO_SIT,47,23
7,SIT_TO_STAND,23,10
8,SIT_TO_LIE,75,32
9,LIE_TO_SIT,60,25


## 3. Retained models and settings

| Model | Author | Selected parameters | Fine-tuning approach |
|---|---|---|---|
| Logistic regression | Yifei | C=0.1, lbfgs, max_iter=600 | Context/weight ablations, participant CV, transition F1 |
| Linear SVM | Yifei | C=0.01, dual=False, max_iter=2000 | Same matched comparisons |
| Random forest | Yifei | 200 trees, seed 42, n_jobs=2 | Same comparisons; ROS-60 checked separately |
| XGBoost | Yifei | 300 trees, depth 6, learning rate 0.138, subsample 0.616, column sample 0.847, gamma 0.00075, minimum child weight 7, seed 42 | Same matched comparisons |
| MLP + posture memory | Tianfeng | 256–128–64; ReLU; dropout 0.3/0.3/0.2; Adam lr=0.001, weight decay=0.0001; batch 128; 15 epochs | 19 settings, then a six-setting shortlist; 93 participant-validation fits |

All retained models use previous/current/next windows, training-fitted scaling,
all training rows, mild weighting, no PCA and no oversampling in the final fit.
Yifei normalises sample weights to mean 1. Tianfeng's NN divides each batch's
weighted loss by its sum of weights, then applies eight-window posture memory.
The selected random forest remains the **no-ROS** variant from the completed
shared comparison; separate ROS checks remain available under optional tuning.

Following-window context requires offline or delayed predictions. Context resets
at participant boundaries. Chronological order is assumed; the files do not mark
every within-person recording boundary. These are complete pipeline comparisons.

## 4. Yifei — context, weights and the four retained classifiers

`add_context` adds neighbours before shuffling; `prepare` fits scaling and optional
PCA on training rows; `predict` trains one of the four classical classifiers.
Their model parameters and weighting remain as implemented in Yifei's notebook.
The MLP reference is excluded because Tianfeng's MLP + memory had higher
transition F1. The optional participant-CV functions retain Yifei's balancing
procedure, with validation rows left naturally imbalanced.

In [3]:
LABELS = np.arange(1, 13)
TRANSITION_LABELS = np.arange(7, 13)

def add_context(x, groups, mode="single"):
    if mode == "single":
        return x
    previous = np.vstack((x[:1], x[:-1])).copy()
    previous[np.flatnonzero(groups[1:] != groups[:-1]) + 1] = x[np.flatnonzero(groups[1:] != groups[:-1]) + 1]
    if mode == "causal":
        return np.hstack((x, previous))
    if mode != "bidirectional":
        raise ValueError(f"Unknown context mode: {mode}")
    following = np.vstack((x[1:], x[-1:])).copy()
    boundary = np.flatnonzero(groups[:-1] != groups[1:])
    following[boundary] = x[boundary]
    return np.hstack((previous, x, following))


def class_weights(y, exponent=0.5):
    counts = np.bincount(y, minlength=13)
    sample_weight = counts[y].astype(float) ** (-exponent)
    return sample_weight / sample_weight.mean()


def oversample_indices(y, target, seed=42):
    rng = np.random.default_rng(seed)
    indices = [np.arange(len(y))]
    for label in TRANSITION_LABELS:
        found = np.flatnonzero(y == label)
        if len(found) < target:
            indices.append(rng.choice(found, size=target - len(found), replace=True))
    return np.concatenate(indices)


def scores(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    return {
        "transition_f1": float(f1[6:].mean()),
        "transition_recall": float(recall[6:].mean()),
        "common_f1": float(f1[:6].mean()),
        "macro_f1": float(f1.mean()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "per_class_f1": f1,
        "per_class_recall": recall,
        "per_class_precision": precision,
        "support": support,
    }

In [4]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC


def prepare(x_train, x_eval, representation="raw"):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(x_train)
    eval_scaled = scaler.transform(x_eval)
    if representation == "pca200":
        pca = PCA(n_components=200, svd_solver="randomized", random_state=42)
        train_scaled = pca.fit_transform(train_scaled)
        eval_scaled = pca.transform(eval_scaled)
    return train_scaled.astype(np.float32), eval_scaled.astype(np.float32)


def predict(model_name, x_train, y_train, x_eval, sample_weight=None):
    if model_name == "logreg":
        model = LogisticRegression(C=0.1, solver="lbfgs", max_iter=600)
    elif model_name == "linear_svm":
        model = LinearSVC(C=0.01, dual=False, max_iter=2000)
    elif model_name == "random_forest":
        model = RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=42)
    elif model_name == "xgboost":
        from xgboost import XGBClassifier
        model = XGBClassifier(
            objective="multi:softprob", num_class=12, tree_method="hist", n_jobs=2,
            n_estimators=300, max_depth=6, learning_rate=0.138,
            subsample=0.616, colsample_bytree=0.847, gamma=0.00075,
            min_child_weight=7, random_state=42,
        )
        model.fit(x_train, y_train - 1, sample_weight=sample_weight)
        return model.predict(x_eval) + 1
    else:
        raise ValueError(model_name)
    model.fit(x_train, y_train, sample_weight=sample_weight)
    return model.predict(x_eval)


def evaluate_cv(model_name="logreg", context="bidirectional", weight_exponent=0.5,
                oversample_target=0, representation="raw", seed=42):
    features = add_context(x_train, groups_train, context)
    splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    rows = []
    for fold, (tr, va) in enumerate(splitter.split(features, y_train, groups_train), 1):
        assert set(groups_train[tr]).isdisjoint(groups_train[va])
        fold_x, fold_y = features[tr], y_train[tr]
        sample_weight = class_weights(fold_y, weight_exponent) if weight_exponent else None
        if oversample_target:
            idx = oversample_indices(fold_y, oversample_target, seed=42 + fold)
            fold_x, fold_y = fold_x[idx], fold_y[idx]
            if sample_weight is not None:
                sample_weight = sample_weight[idx]
        fit_x, val_x = prepare(fold_x, features[va], representation)
        pred = predict(model_name, fit_x, fold_y, val_x, sample_weight)
        rows.append({"fold": fold, **scores(y_train[va], pred)})
    return pd.DataFrame(rows)


def evaluate_test(model_name="logreg", context="bidirectional", weight_exponent=0.5,
                  oversample_target=0, representation="raw"):
    train_features = add_context(x_train, groups_train, context)
    test_features = add_context(x_test, groups_test, context)
    fit_y = y_train.copy()
    sample_weight = class_weights(fit_y, weight_exponent) if weight_exponent else None
    if oversample_target:
        idx = oversample_indices(fit_y, oversample_target, seed=42)
        train_features, fit_y = train_features[idx], fit_y[idx]
        if sample_weight is not None:
            sample_weight = sample_weight[idx]
    fit_x, eval_x = prepare(train_features, test_features, representation)
    pred = predict(model_name, fit_x, fit_y, eval_x, sample_weight)
    return scores(y_test + 1, pred)

## 5. Train the four retained classical models

The selected pipeline uses three-window context, mild weighting and all training
rows. Fit each classifier once. Keep Yifei's selected parameters, with no PCA or
oversampling in this final comparison.

In [5]:
x_train, y_train, groups_train = X, y + 1, subject
x_test, groups_test = X_test, test_subject
yifei_train_context = add_context(x_train, groups_train, 'bidirectional')
yifei_test_context = add_context(x_test, groups_test, 'bidirectional')
yifei_fit_x, yifei_eval_x = prepare(yifei_train_context, yifei_test_context)
yifei_weights = class_weights(y_train, exponent=0.5)
yifei_models = {
    'logreg': 'Logistic regression', 'linear_svm': 'Linear SVM',
    'random_forest': 'Random forest', 'xgboost': 'XGBoost',
}
for model_name, title in yifei_models.items():
    predicted = predict(model_name, yifei_fit_x, y_train, yifei_eval_x,
                        sample_weight=yifei_weights)
    predictions_by_model['Yifei | ' + title] = predicted - 1
    print('Yifei:', title, 'fitted.')

Yifei: Logistic regression fitted.
Yifei: Linear SVM fitted.
Yifei: Random forest fitted.
Yifei: XGBoost fitted.


## 6. Tianfeng — selected NN and training loop

Use the same three-window representation, all rows, no PCA and no oversampling.
The architecture and weighted-loss calculation below are unchanged from the
selected NN. A batch loss divides by its **sum of weights**. Fifteen fixed
epochs came from the median best epoch across nine validation folds; the final
fit does not early-stop using test labels.

In [6]:
SEED = 42
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
WEIGHT_POWER = 0.5
BATCH_SIZE = 128
EPOCHS = 15
scaler = StandardScaler()
train_x = scaler.fit_transform(yifei_train_context).astype(np.float32)
train_y = y

In [7]:
torch.manual_seed(SEED)
model = nn.Sequential(
    nn.Linear(train_x.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 12),
)
for layer in model:
    if isinstance(layer, nn.Linear):
        if layer.out_features == 12:
            nn.init.xavier_uniform_(layer.weight)
        else:
            nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
        nn.init.zeros_(layer.bias)

counts = np.bincount(train_y, minlength=12)
weights = torch.tensor((len(train_y) / (12 * counts)) ** WEIGHT_POWER, dtype=torch.float32)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loader = DataLoader(TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y)),
                    batch_size=BATCH_SIZE, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))

In [8]:
train_losses = []
batch_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        losses = F.cross_entropy(logits, batch_y, reduction='none')
        loss = (losses * weights[batch_y]).sum() / weights[batch_y].sum()
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        target = torch.from_numpy(train_y)
        logits = model(torch.from_numpy(train_x))
        losses = F.cross_entropy(logits, target, reduction='none')
        train_loss = (losses * weights[target]).sum() / weights[target].sum()
    train_losses.append(train_loss.item())
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f'Epoch {epoch:2d}/{EPOCHS}: training loss {train_loss.item():.4f}')

Epoch  1/15: training loss 0.3762
Epoch  5/15: training loss 0.0608
Epoch 10/15: training loss 0.0254
Epoch 15/15: training loss 0.0175


## 7. Tianfeng — predict and apply posture memory

The plain NN produces scores first. The eight-window rule can only swap
sit-to-lie and stand-to-lie, based on earlier sitting/standing NN scores from
the same participant. It never reads true labels. Its evidence threshold is
0.75 score mass, not a calibrated 75% confidence. The plain NN row in the final
table shows what this postprocessing changes.

In [9]:
def memory_predictions(probabilities, people):
    prediction = probabilities.argmax(axis=1)
    gate = np.isin(prediction, [8, 10])
    mass = probabilities[:, [8, 10]].sum(axis=1)
    score = np.divide(probabilities[:, 10], mass,
                      out=np.full(len(people), 0.5), where=mass > 0)
    start = 0
    for row in range(len(people)):
        if row == 0 or people[row] != people[row - 1]:
            start = row
        past = probabilities[max(start, row - 8):row, [3, 4]].sum(axis=0)
        if past.sum() >= 0.75:
            score[row] = past[1] / past.sum()
    prediction[gate] = np.where(score[gate] > 0.5, 10, 8)
    return prediction

test_x = scaler.transform(yifei_test_context).astype(np.float32)
model.eval()
with torch.no_grad():
    probabilities = model(torch.from_numpy(test_x)).softmax(dim=1).numpy()
baseline = probabilities.argmax(axis=1)
final_prediction = memory_predictions(probabilities, test_subject)
predictions_by_model['Tianfeng | Context MLP + memory'] = final_prediction
print('Tianfeng: posture memory changed', np.sum(baseline != final_prediction), 'predictions.')

Tianfeng: posture memory changed 15 predictions.


## 8. One shared metrics table — one row per model family

Every row uses the same 3,162 test windows and natural class imbalance. The
**Author** column identifies the retained implementation; MLP refers to
Tianfeng's three-window NN with posture memory. No duplicate family is shown.

Accuracy is the fraction of correct windows. Macro precision, recall and F1 give
each class equal weight; weighted F1 weights by support. Transition F1/recall
average the six rare transitions; common F1 averages the six common activities.
All values are fractions, so 0.95 means 95%.

The retained variants were chosen from the earlier observed transition F1 scores.
This is development-test selection, not an untouched final evaluation or a
validation-only model-selection claim. The original validation methods differ.

In [10]:
rows = []
per_class_results = {}
for title, predicted in predictions_by_model.items():
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, predicted, labels=range(12), zero_division=0)
    per_class_results[title] = (precision, recall, f1, support)
    owner, model_title = title.split(' | ')
    rows.append({
        'Model': 'MLP + memory' if owner == 'Tianfeng' else model_title,
        'Author': owner,
        'Accuracy': np.mean(y_test == predicted),
        'Macro precision': precision.mean(), 'Macro recall': recall.mean(),
        'Macro F1': f1.mean(), 'Weighted F1': np.average(f1, weights=support),
        'Transition F1': f1[6:].mean(), 'Transition recall': recall[6:].mean(),
        'Common F1': f1[:6].mean(),
    })
metrics_table = pd.DataFrame(rows)
display(metrics_table.round(4))

,Model,Author,Accuracy,Macro precision,Macro recall,Macro F1,Weighted F1,Transition F1,Transition recall,Common F1
0,Logistic regression,Yifei,0.9538,0.9326,0.9257,0.9275,0.9538,0.8966,0.8934,0.9585
1,Linear SVM,Yifei,0.9630,0.9289,0.9235,0.9237,0.9628,0.8778,0.8773,0.9697
2,Random forest,Yifei,0.9200,0.9032,0.9005,0.9003,0.9196,0.8788,0.8800,0.9219
3,XGBoost,Yifei,0.9535,0.9209,0.9106,0.9141,0.9532,0.8701,0.8628,0.9581
4,MLP + memory,Tianfeng,0.9560,0.9547,0.9515,0.9511,0.9561,0.9449,0.9471,0.9572


## 9. Inspect the six rare classes

The same five retained models appear below, with their authors in the row labels.
Each transition has its own F1 score, so a strong average cannot hide a weak
sit-to-lie or stand-to-lie result. Test support shows the number of examples.

In [11]:
transition_table = pd.DataFrame(
    {title: result[2][6:] for title, result in per_class_results.items()},
    index=names[6:]).T
transition_table.index.name = 'Author | model'
display(transition_table.round(4))
print('Test support:', dict(zip(names[6:], np.bincount(y_test, minlength=12)[6:].tolist())))

Test support: {'STAND_TO_SIT': 23, 'SIT_TO_STAND': 10, 'SIT_TO_LIE': 32, 'LIE_TO_SIT': 25, 'STAND_TO_LIE': 49, 'LIE_TO_STAND': 27}


,STAND_TO_SIT,SIT_TO_STAND,SIT_TO_LIE,LIE_TO_SIT,STAND_TO_LIE,LIE_TO_STAND
Author | model,,,,,,
Yifei | Logistic regression,0.9778,0.9524,0.8125,0.9057,0.8333,0.8980
Yifei | Linear SVM,0.9778,0.9524,0.8116,0.8679,0.8000,0.8571
Yifei | Random forest,0.8696,1.0000,0.7826,0.9231,0.7957,0.9020
Yifei | XGBoost,0.8511,1.0000,0.8358,0.8846,0.8090,0.8400
Tianfeng | Context MLP + memory,0.9565,1.0000,0.9688,0.9091,0.9600,0.8750


## 10. Per-class details for any model

Change `selected_model` to any row name above. Per-class accuracy is recall:
correct predictions divided by that class's support. It is shown as a percentage
rather than treating one-vs-rest accuracy as evidence of minority performance.
The confusion matrix has actual classes in rows and predicted classes in columns.

In [12]:
selected_model = 'Tianfeng | Context MLP + memory'
p, r, f, n = per_class_results[selected_model]
cm = confusion_matrix(y_test, predictions_by_model[selected_model], labels=range(12))
display(pd.DataFrame({'Activity': names, 'Precision': p, 'Recall': r, 'F1': f,
                      'Class accuracy %': 100 * r, 'Correct': np.diag(cm), 'Support': n}).round(4))
display(pd.DataFrame(cm, index=names, columns=names))

,Activity,Precision,Recall,F1,Class accuracy %,Correct,Support
0,WALKING,0.9374,0.9960,0.9658,99.5968,494,496
1,WALKING_UPSTAIRS,0.9701,0.9660,0.9681,96.6030,455,471
2,WALKING_DOWNSTAIRS,0.9924,0.9286,0.9594,92.8571,390,420
3,SITTING,0.9493,0.9213,0.9351,92.1260,468,508
4,STANDING,0.9077,0.9550,0.9308,95.5036,531,556
5,LAYING,1.0000,0.9688,0.9842,96.8807,528,545
6,STAND_TO_SIT,0.9565,0.9565,0.9565,95.6522,22,23
7,SIT_TO_STAND,1.0000,1.0000,1.0000,100.0000,10,10
8,SIT_TO_LIE,0.9688,0.9688,0.9688,96.8750,31,32
9,LIE_TO_SIT,0.8333,1.0000,0.9091,100.0000,25,25


,WALKING,WALKING_UPSTAIRS,WALKING_DOWNSTAIRS,SITTING,STANDING,LAYING,STAND_TO_SIT,SIT_TO_STAND,SIT_TO_LIE,LIE_TO_SIT,STAND_TO_LIE,LIE_TO_STAND
WALKING,494,0,2,0,0,0,0,0,0,0,0,0
WALKING_UPSTAIRS,15,455,1,0,0,0,0,0,0,0,0,0
WALKING_DOWNSTAIRS,17,13,390,0,0,0,0,0,0,0,0,0
SITTING,0,1,0,468,37,0,1,0,1,0,0,0
STANDING,0,0,0,24,531,0,0,0,0,0,1,0
LAYING,0,0,0,0,16,528,0,0,0,0,1,0
STAND_TO_SIT,0,0,0,1,0,0,22,0,0,0,0,0
SIT_TO_STAND,0,0,0,0,0,0,0,10,0,0,0,0
SIT_TO_LIE,0,0,0,0,0,0,0,0,31,0,1,0
LIE_TO_SIT,0,0,0,0,0,0,0,0,0,25,0,0


## 11. Optional tuning — Yifei's retained classifiers

Keep the four selected classifiers' fixed parameters while repeating the original
single/context × no/mild weighting comparisons, ROS targets 60/100 and PCA-200.
Extra participant-split seeds 123/2026 confirm the LR/SVM context-plus-weighting
comparison and the RF ROS-60 check. These are the original checks for the retained
families; the excluded MLP reference and its epoch sweep are omitted.

Scaling/PCA use training-fold rows only. Duplicated training rows inherit their
original weights. Validation rows are never resampled. The primary metric stays
transition macro-F1, with common-class F1 as a check.

In [13]:
RUN_YIFEI_TUNING = False

def run_yifei_tuning():
    checks = []
    for model_name in yifei_models:
        for context in ['single', 'bidirectional']:
            for weight in [0.0, 0.5]:
                checks.append((model_name, context, weight, 0, 'raw', 42))
        for target in [60, 100]:
            checks.append((model_name, 'bidirectional', 0.5, target, 'raw', 42))
        checks.append((model_name, 'bidirectional', 0.5, 0, 'pca200', 42))
    for seed in [123, 2026]:
        for model_name in ['logreg', 'linear_svm', 'random_forest']:
            checks.append((model_name, 'bidirectional', 0.5, 0, 'raw', seed))
        checks.append(('random_forest', 'bidirectional', 0.5, 60, 'raw', seed))

    summaries = []
    for model_name, context, weight, ros, representation, seed in checks:
        folds = evaluate_cv(model_name, context, weight, ros, representation, seed)
        summaries.append({'Model': model_name, 'Context': context, 'Weight power': weight,
                          'ROS target': ros, 'Representation': representation,
                          'Split seed': seed,
                          'Transition F1': folds.transition_f1.mean(),
                          'Common F1': folds.common_f1.mean()})
        print(model_name, context, weight, ros, representation, seed,
              '| Transition F1:', round(folds.transition_f1.mean(), 4))
    return pd.DataFrame(summaries)

if RUN_YIFEI_TUNING:
    yifei_tuning_table = run_yifei_tuning()
    display(yifei_tuning_table.round(4))

## 12. Optional tuning — Tianfeng

This retains the full 93-fit study: 19 configurations × three participant folds,
then the reference and five shortlisted settings on two extra split/training
seed schedules. It varies context, PCA, common-class cap, weighting and random
oversampling, keeping architecture and optimiser fixed. New trials stop on
transition F1, with maximum 100 epochs and patience 12. The old reference stops
on all-class F1. Keep the best weights, then compare memory on and off.

The final choice must keep common-class F1 within 0.01 of the reference.
Offline context is allowed. The median of the chosen setting's nine best epochs
sets the final training duration. The selected previous study chose three
windows, no PCA/cap/ROS, mild weights, memory, and 15 epochs. This cell reruns
selection using **training participants only**, even though test results appear
earlier in the notebook. It does not feed those test scores into selection or
change the frozen main comparison automatically.

In [14]:
RUN_TIANFENG_TUNING = False

def run_tianfeng_tuning():
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.0001
    BATCH_SIZE = 128
    MAX_EPOCHS = 100
    PATIENCE = 12
    ALLOW_NEXT_WINDOW = True
    views = {}
    rows = np.arange(len(X))
    for name, offsets in [('single', [0]), ('past2', [-1, 0]),
                          ('past3', [-2, -1, 0]), ('offline3', [-1, 0, 1])]:
        blocks = []
        for offset in offsets:
            neighbor = np.clip(rows + offset, 0, len(X) - 1)
            boundary = subject[neighbor] != subject
            neighbor[boundary] = rows[boundary]
            blocks.append(X[neighbor])
        views[name] = np.hstack(blocks)
    print('Input dimensions:', {name: values.shape[1] for name, values in views.items()})


    def make_model(inputs):
        model = nn.Sequential(
            nn.Linear(inputs, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 12),
        )
        for layer in model:
            if isinstance(layer, nn.Linear):
                if layer.out_features == 12:
                    nn.init.xavier_uniform_(layer.weight)
                else:
                    nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)
        return model


    def memory_predictions(probabilities, people):
        prediction = probabilities.argmax(axis=1)
        gate = np.isin(prediction, [8, 10])
        mass = probabilities[:, [8, 10]].sum(axis=1)
        score = np.divide(probabilities[:, 10], mass, out=np.full(len(people), 0.5), where=mass > 0)
        start = 0
        for row in range(len(people)):
            if row == 0 or people[row] != people[row - 1]:
                start = row
            past = probabilities[max(start, row - 8):row, [3, 4]].sum(axis=0)
            if past.sum() >= 0.75:
                score[row] = past[1] / past.sum()
        prediction[gate] = np.where(score[gate] > 0.5, 10, 8)
        return prediction


    def fit_fold(config, tr, va, fold, split_seed):
        name, context, components, limit, exponent, ros, stopping = config
        seed = split_seed + fold
        features = views[context]
        rng = np.random.default_rng(seed)
        selected = []
        for activity in range(12):
            rows = np.flatnonzero(y[tr] == activity)
            if activity < 6 and limit and len(rows) > limit:
                rows = rng.choice(rows, limit, replace=False)
            selected.extend(rows)
        sampled = tr[np.sort(selected)]
        counts = np.bincount(y[sampled], minlength=12)
        weights = torch.tensor((len(sampled) / (12 * counts)) ** exponent, dtype=torch.float32)
        if ros:
            extra = []
            for activity in range(6, 12):
                found = sampled[y[sampled] == activity]
                if len(found) < ros:
                    extra.extend(rng.choice(found, ros - len(found), replace=True))
            sampled = np.concatenate((sampled, np.asarray(extra, dtype=int)))
        scaler = StandardScaler()
        train_x = scaler.fit_transform(features[sampled]).astype(np.float32)
        val_x = scaler.transform(features[va]).astype(np.float32)
        if components:
            pca = PCA(n_components=components, svd_solver='randomized', random_state=seed)
            train_x = pca.fit_transform(train_x).astype(np.float32)
            val_x = pca.transform(val_x).astype(np.float32)
        train_y = y[sampled]
        tx, ty, vx = torch.from_numpy(train_x), torch.from_numpy(train_y), torch.from_numpy(val_x)
        vy = torch.from_numpy(y[va])
        torch.manual_seed(seed)
        model = make_model(train_x.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        loader = DataLoader(TensorDataset(tx, ty), batch_size=BATCH_SIZE, shuffle=True,
                            generator=torch.Generator().manual_seed(seed))
        best_score = -1.0
        best_epoch = 0
        history = []
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            for bx, by in loader:
                optimizer.zero_grad()
                logits = model(bx)
                losses = F.cross_entropy(logits, by, reduction='none')
                loss = (losses * weights[by]).sum() / weights[by].sum()
                loss.backward()
                optimizer.step()
            model.eval()
            with torch.no_grad():
                train_logits = model(tx)
                losses = F.cross_entropy(train_logits, ty, reduction='none')
                train_loss = (losses * weights[ty]).sum() / weights[ty].sum()
                val_logits = model(vx)
                losses = F.cross_entropy(val_logits, vy, reduction='none')
                val_loss = (losses * weights[vy]).sum() / weights[vy].sum()
                f = precision_recall_fscore_support(y[va], val_logits.argmax(1).numpy(), labels=range(12), zero_division=0)[2]
            score = float(f.mean() if stopping == 'all' else f[6:].mean())
            history.append({'epoch': epoch, 'train_loss': train_loss.item(), 'validation_loss': val_loss.item(),
                            'transition_f1': float(f[6:].mean()), 'macro_f1': float(f.mean())})
            if score > best_score + 1e-6:
                best_score, best_epoch = score, epoch
                best_weights = deepcopy(model.state_dict())
            if epoch - best_epoch >= PATIENCE:
                break
        model.load_state_dict(best_weights)
        model.eval()
        with torch.no_grad():
            probabilities = model(vx).softmax(1).numpy()
        result = {'config': config, 'split_seed': split_seed, 'fold': fold,
                  'best_epoch': best_epoch, 'stop_epoch': epoch, 'history': history}
        for method, prediction in [('plain', probabilities.argmax(1)),
                                   ('memory', memory_predictions(probabilities, subject[va]))]:
            p, r, f, n = precision_recall_fscore_support(
                y[va], prediction, labels=range(12), zero_division=0)
            result[method] = {'transition_f1': float(f[6:].mean()),
                              'common_f1': float(f[:6].mean()), 'macro_f1': float(f.mean()),
                              'accuracy': float(np.mean(y[va] == prediction)),
                              'transition_recall': float(r[6:].mean()),
                              'class_f1': f.tolist(), 'class_recall': r.tolist()}
        return result


    trials = [
        ('current', 'single', 200, 300, 1.0, 0, 'all'),
        ('transition_stop', 'single', 200, 300, 1.0, 0, 'transition'),
        ('mild_weights_only', 'single', 200, 300, 0.5, 0, 'transition'),
        ('all_rows_only', 'single', 200, 0, 1.0, 0, 'transition'),
        ('no_pca_only', 'single', 0, 300, 1.0, 0, 'transition'),
        ('past2_only', 'past2', 200, 300, 1.0, 0, 'transition'),
        ('past3_only', 'past3', 200, 300, 1.0, 0, 'transition'),
        ('offline3_only', 'offline3', 200, 300, 1.0, 0, 'transition'),
        ('past2_full_raw_mild', 'past2', 0, 0, 0.5, 0, 'transition'),
        ('past3_full_raw_mild', 'past3', 0, 0, 0.5, 0, 'transition'),
        ('offline3_full_raw_mild', 'offline3', 0, 0, 0.5, 0, 'transition'),
        ('offline_raw_unweighted', 'offline3', 0, 0, 0.0, 0, 'transition'),
        ('offline_raw_inverse', 'offline3', 0, 0, 1.0, 0, 'transition'),
        ('offline_all_pca_mild', 'offline3', 200, 0, 0.5, 0, 'transition'),
        ('offline_cap_raw_mild', 'offline3', 0, 300, 0.5, 0, 'transition'),
        ('offline_ros60', 'offline3', 0, 0, 0.5, 60, 'transition'),
        ('offline_ros100', 'offline3', 0, 0, 0.5, 100, 'transition'),
        ('past2_raw_unweighted', 'past2', 0, 0, 0.0, 0, 'transition'),
        ('past2_raw_inverse', 'past2', 0, 0, 1.0, 0, 'transition'),
    ]


    results = []
    folds = list(StratifiedGroupKFold(3, shuffle=True, random_state=42).split(X, y, subject))
    for trial in trials:
        for fold, (tr, va) in enumerate(folds, 1):
            result = fit_fold(trial, tr, va, fold, split_seed=42)
            results.append(result)
            print(f'{trial[0]}, fold {fold}: NN={result["plain"]["transition_f1"]:.4f}, '
                  f'+memory={result["memory"]["transition_f1"]:.4f}, epoch={result["best_epoch"]}')


    print(f'{"Trial":27} {"NN transition F1":>17} {"+memory F1":>12} {"Common F1":>11}')
    for trial in trials:
        group = [r for r in results if r['config'][0] == trial[0]]
        plain = np.mean([r['plain']['transition_f1'] for r in group])
        memory_score = np.mean([r['memory']['transition_f1'] for r in group])
        common = np.mean([r['memory']['common_f1'] for r in group])
        print(f'{trial[0]:27} {plain:17.4f} {memory_score:12.4f} {common:11.4f}')


    ranked = []
    for trial in trials:
        group = [r for r in results if r['config'][0] == trial[0]]
        score = max(np.mean([r[method]['transition_f1'] for r in group])
                    for method in ['plain', 'memory'])
        ranked.append((score, trial))
    ranked.sort(key=lambda item: item[0], reverse=True)
    offline = [trial for score, trial in ranked if trial[1] == 'offline3'][:4]
    past_only = [trial for score, trial in ranked if trial[1] != 'offline3' and trial[0] != 'current'][:1]
    shortlist = [trials[0]] + offline + past_only

    for trial in shortlist:
        for split_seed in [123, 2026]:
            folds = list(StratifiedGroupKFold(3, shuffle=True, random_state=split_seed).split(X, y, subject))
            for fold, (tr, va) in enumerate(folds, 1):
                result = fit_fold(trial, tr, va, fold, split_seed)
                results.append(result)
                print(f'{trial[0]}, seed {split_seed}, fold {fold}: '
                      f'+memory F1={result["memory"]["transition_f1"]:.4f}, epoch={result["best_epoch"]}')


    reference = [r for r in results if r['config'][0] == 'current']
    minimum_common = np.mean([r['memory']['common_f1'] for r in reference]) - 0.01
    choices = []
    print(f'{"Shortlist":27} {"Method":>8} {"Transition F1":>14} {"Fold SD":>9} {"Common F1":>11}')
    for trial in shortlist:
        group = [r for r in results if r['config'][0] == trial[0]]
        for method in ['plain', 'memory']:
            values = [r[method]['transition_f1'] for r in group]
            mean = float(np.mean(values))
            common = float(np.mean([r[method]['common_f1'] for r in group]))
            print(f'{trial[0]:27} {method:>8} {mean:14.4f} {np.std(values, ddof=1):9.4f} {common:11.4f}')
            if common >= minimum_common and (ALLOW_NEXT_WINDOW or trial[1] != 'offline3'):
                choices.append((mean, trial, method))
    best_score, best_trial, best_method = max(choices, key=lambda item: item[0])
    chosen = [r for r in results if r['config'][0] == best_trial[0]]
    final_epochs = int(np.median([r['best_epoch'] for r in chosen]))
    print('\nSelected:', best_trial[0], '| Method:', best_method)
    print('Best epochs:', [r['best_epoch'] for r in chosen], '| Final duration:', final_epochs)


    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for column, result in enumerate([r for r in chosen if r['split_seed'] == 42]):
        history = result['history']
        epochs = [row['epoch'] for row in history]
        axes[0, column].plot(epochs, [row['train_loss'] for row in history], label='Training')
        axes[0, column].plot(epochs, [row['validation_loss'] for row in history], label='Validation')
        axes[0, column].axvline(result['best_epoch'], color='green', linestyle='--')
        axes[0, column].set(title=f'Fold {column + 1}', xlabel='Epoch', ylabel='Weighted loss')
        axes[0, column].legend()
        axes[1, column].plot(epochs, [row['transition_f1'] for row in history])
        axes[1, column].axvline(result['best_epoch'], color='green', linestyle='--')
        axes[1, column].set(xlabel='Epoch', ylabel='Validation transition F1')
    fig.tight_layout()
    plt.show()


    old_f1 = np.mean([r['memory']['class_f1'] for r in reference], axis=0)
    new_f1 = np.mean([r[best_method]['class_f1'] for r in chosen], axis=0)
    print(f'{"Transition":22} {"Previous F1":>12} {"Selected F1":>12} {"Change":>9}')
    for activity in range(6, 12):
        print(f'{names[activity]:22} {old_f1[activity]:12.4f} {new_f1[activity]:12.4f}'
              f' {new_f1[activity] - old_f1[activity]:+9.4f}')

    return best_trial, best_method, final_epochs, results

if RUN_TIANFENG_TUNING:
    tianfeng_selection = run_tianfeng_tuning()

## Reading the comparison

The main run contains one retained variant per family, with the author's name.
Their original preprocessing, weighting and hyperparameters stay intact. Original
teammate files are unchanged. Tuning remains optional and uses training
participants; rerunning it does not silently replace the retained settings.

The family representatives were chosen using the earlier shared development-test
results. Their table is calculated again here, rather than copied from historical
numbers. This reporting choice cannot establish superiority on unseen data; a
new held-out participant set would be needed to assess the selected collection.